# Auditoría de Calidad de Datos — NovaMarket

Autor: Daniel Pallarés, Sofia Arbelaez, Juan Sanchez
Dataset: NovaMarket_datos_crudos.csv (640,000 registros, 20 columnas)
Fecha de referencia de la auditoría: 2026-08-04
Objetivo: cuantificar, con conteos exactos, la confiabilidad del dataset crudo de NovaMarket a lo
largo de las 6 dimensiones de calidad de datos: completitud, exactitud, consistencia, validez,
unicidad y oportunidad. Este notebook es la base del Informe de Calidad de Datos de la Misión 1.

Nota de escala: este dataset tiene 640,000 filas, por lo que todo el código de esta auditoría está
escrito de forma vectorizada (sin bucles fila por fila ni try/except por registro) para que se
ejecute en segundos y no en minutos.

La sección final construye el catálogo de problemas
(S03_PD_Pallares_CatalogoProblemas.csv), ordenado de mayor a menor severidad.

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

df = pd.read_csv('NovaMarket_datos_crudos.csv')
N = len(df)
HOY = pd.Timestamp('2026-08-04')  # fecha de referencia de la auditoria

print(f"Filas: {N:,} | Columnas: {df.shape[1]}")
df.head()

Filas: 640,000 | Columnas: 20


,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,unidades_vendidas,monto_compra,edad_cliente,correo_cliente,nivel_satisfaccion,nivel_lealtad,comentario_cliente,peso_pedido_kg,fecha_actualizacion_stock,fuga_cliente
0,P505195,C45650,15/11/2025,Tienda,Efectivo,bogota,1100103,Moda,Chaqueta impermeable,28700.0,1,28700.0,32,cliente405195@novamarket.com.co,Bajo,Platino,El domiciliario fue muy amable,14.9,2026-04-17,0
1,P677836,C84260,2026-03-05,Web,Efectivo,MZL,1700101,Electrónica,Parlante bluetooth,195700.0,2,391400.0,73,cliente577836@yahoo.es,Medio,Bronce,El producto llegó dañado,12.9,2026-06-10,0
2,P387869,C164166,2025-03-13,Tienda,PayPal,Cúcuta,5400101,Juguetería,Rompecabezas 1000 piezas,597600.0,-3,-1792800.0,32,cliente287869@novamarket.com.co,Bajo,Oro,El domiciliario fue muy amable,14.3,2026-02-26,0
3,P481069,C98406,27/06/2025,App,transferencia,Cucuta,5400103,deportes,Colchoneta de yoga,322900.0,1,322900.0,45,cliente381069@gmail.com,Medio,Platino,Rápido y sin problemas,9.5,2026-05-14,0
4,P247920,C163018,01/05/2025,Web,Efectivo,Bucaramanga,6800112,Hogar,Juego de sábanas,222400.0,1,222400.0,60,cliente147920@novamarket.com.co,Alto,Plata,NaN,12.3,2026-07-13,0


## 1. Completitud

Mide qué proporción de valores esperados realmente están presentes. Se calcula el % de faltantes
para las 20 columnas y se detallan las columnas con valores nulos.

In [2]:
faltantes = pd.DataFrame({
    'faltantes': df.isna().sum(),
    '%_faltantes': (df.isna().mean() * 100).round(2),
})
faltantes = faltantes[faltantes['faltantes'] > 0].sort_values('%_faltantes', ascending=False)
faltantes

,faltantes,%_faltantes
correo_cliente,105312,16.46
comentario_cliente,90806,14.19
nivel_satisfaccion,46469,7.26
ciudad_tienda,29928,4.68
precio_unitario,21640,3.38
monto_compra,21640,3.38


In [3]:
# precio_unitario y monto_compra: verificar si faltan siempre en las mismas filas
mismas_filas = (df['precio_unitario'].isna() & df['monto_compra'].isna()).sum()
solo_precio = (df['precio_unitario'].isna() & df['monto_compra'].notna()).sum()
solo_monto = (df['precio_unitario'].notna() & df['monto_compra'].isna()).sum()
print(f"precio_unitario y monto_compra faltantes en las mismas filas: {mismas_filas}")
print(f"solo precio_unitario faltante: {solo_precio} | solo monto_compra faltante: {solo_monto}")

precio_unitario y monto_compra faltantes en las mismas filas: 21640
solo precio_unitario faltante: 0 | solo monto_compra faltante: 0


Hallazgos — Completitud (sobre 640,000 registros):

- correo_cliente: 105,312 registros faltantes (16.46%). Es la columna con más faltantes; sin ella
  no hay forma de contactar directamente a ese cliente.
- comentario_cliente: 90,806 registros faltantes (14.19%). Es un campo de texto libre y opcional
  (no todo cliente deja comentario), por lo que este faltante es esperable y de menor criticidad.
- nivel_satisfaccion: 46,469 registros faltantes (7.26%). Impide calcular métricas de satisfacción
  para ese subconjunto.
- ciudad_tienda: 29,928 registros faltantes (4.68%). Impide segmentación geográfica y logística.
- precio_unitario y monto_compra faltan exactamente en las mismas 21,640 filas (3.38%): tiene
  sentido, porque monto_compra se deriva de precio_unitario, así que cuando falta uno falta el otro.
- Las 13 columnas restantes no tienen valores faltantes (0.00%).

## 2. Exactitud

Mide si los valores presentes son correctos y físicamente/lógicamente posibles, más allá de si
están o no. Se evalúan precio_unitario, unidades_vendidas, edad_cliente, peso_pedido_kg, una regla
de negocio derivada (monto_compra = precio_unitario × unidades_vendidas) y una regla de dominio
propia sobre fecha_pedido (una compra no puede registrarse en una fecha posterior a hoy).

In [4]:
precio_negativo = (df['precio_unitario'] < 0).sum()
print(f"precio_unitario negativo: {precio_negativo:,} ({precio_negativo/N*100:.2f}%)")
print(df['precio_unitario'].describe())

precio_unitario negativo: 19,629 (3.07%)
count    6.183600e+05
mean     5.498868e+06
std      7.003612e+07
min     -8.881000e+05
25%      1.697000e+05
50%      3.056000e+05
75%      4.842000e+05
max      9.990000e+08
Name: precio_unitario, dtype: float64


In [5]:
print("Distribución de unidades_vendidas:")
print(df['unidades_vendidas'].value_counts().sort_index())

unidades_negativas = (df['unidades_vendidas'] < 0).sum()
unidades_cero = (df['unidades_vendidas'] == 0).sum()
unidades_centinela = df['unidades_vendidas'].isin([5000, 9999]).sum()
unidades_invalidas = ((df['unidades_vendidas'] < 0) | (df['unidades_vendidas'] == 0) | df['unidades_vendidas'].isin([5000, 9999])).sum()

print(f"\nunidades_vendidas negativas: {unidades_negativas:,} | en cero: {unidades_cero:,} | centinela (5000/9999): {unidades_centinela:,}")
print(f"unidades_vendidas invalidas en total (negativas, cero o centinela): {unidades_invalidas:,} ({unidades_invalidas/N*100:.2f}%)")

Distribución de unidades_vendidas:
unidades_vendidas
-3         9193
-1         9357
 0         9327
 1       296841
 2       148362
 3        77270
 4        41531
 5        29593
 5000      9185
 9999      9341
Name: count, dtype: int64

unidades_vendidas negativas: 18,550 | en cero: 9,327 | centinela (5000/9999): 18,526
unidades_vendidas invalidas en total (negativas, cero o centinela): 46,403 (7.25%)


In [6]:
# Regla de negocio: monto_compra debe ser exactamente precio_unitario * unidades_vendidas
esperado = df['precio_unitario'] * df['unidades_vendidas']
mask = df['precio_unitario'].notna() & df['monto_compra'].notna()
diferencia = (df.loc[mask, 'monto_compra'] - esperado[mask]).abs()
inconsistente = (diferencia > 1).sum()  # tolerancia de 1 peso por redondeo
print(f"Registros donde monto_compra != precio_unitario * unidades_vendidas: {inconsistente:,} de {mask.sum():,} evaluados")
print("-> la regla se cumple en el 100% de los casos: monto_compra SI es confiable como campo derivado.")

Registros donde monto_compra != precio_unitario * unidades_vendidas: 0 de 618,360 evaluados
-> la regla se cumple en el 100% de los casos: monto_compra SI es confiable como campo derivado.


In [7]:
edad_negativa = (df['edad_cliente'] < 0).sum()
edad_alta = (df['edad_cliente'] > 100).sum()
edad_fuera_rango = edad_negativa + edad_alta
print(f"edad_cliente negativa: {edad_negativa:,} | mayor a 100 anios: {edad_alta:,}")
print(f"edad_cliente fuera de rango fisico en total: {edad_fuera_rango:,} ({edad_fuera_rango/N*100:.2f}%)")

edad_cliente negativa: 9,254 | mayor a 100 anios: 18,575
edad_cliente fuera de rango fisico en total: 27,829 (4.35%)


In [8]:
# peso_pedido_kg: ningun producto individual de NovaMarket deberia pesar mas de 100 kg
print(df['peso_pedido_kg'].describe())
print()
peso_implausible = (df['peso_pedido_kg'] > 100).sum()
print(f"peso_pedido_kg > 100 kg: {peso_implausible:,} ({peso_implausible/N*100:.2f}%)")

# comprobacion: no se concentra en productos "pesados" como bicicletas; aparece parejo en TODOS
# los productos, incluso en una camisa de lino, lo que confirma que es un error, no un caso real
print(df.loc[df['peso_pedido_kg'] > 100, 'producto'].value_counts().head(5))

count    640000.000000
mean         68.466064
std         250.219075
min           8.000000
25%           9.800000
50%          11.700000
75%          13.500000
max        1399.900000
Name: peso_pedido_kg, dtype: float64

peso_pedido_kg > 100 kg: 32,045 (5.01%)
producto
Carro a control remoto    1410
Olla a presión            1399
Teclado mecánico          1390
Camisa de lino            1384
Bicicleta estática        1377
Name: count, dtype: int64


In [9]:
# Regla de dominio propia: una compra no puede registrarse en una fecha futura respecto a la
# fecha de referencia de esta auditoria. Parseo vectorizado con los 2 formatos que usa el dataset
# (nunca se deja que pandas "adivine" el formato).
fecha_pedido_iso = pd.to_datetime(df['fecha_pedido'], format='%Y-%m-%d', errors='coerce')
fecha_pedido_dmy = pd.to_datetime(df['fecha_pedido'], format='%d/%m/%Y', errors='coerce')
fecha_pedido = fecha_pedido_iso.fillna(fecha_pedido_dmy)

fecha_pedido_futura = (fecha_pedido > HOY).sum()
print(f"fecha_pedido posterior a hoy ({HOY.date()}): {fecha_pedido_futura:,} ({fecha_pedido_futura/N*100:.2f}%)")

fecha_pedido posterior a hoy (2026-08-04): 16,485 (2.58%)


Hallazgos — Exactitud (sobre 640,000 registros):

- unidades_vendidas inválidas (negativas, en cero, o con valores "centinela" 5000/9999 que parecen
  códigos de error): 46,403 registros (7.25%).
- precio_unitario negativo (imposible para una venta): 19,629 registros (3.07%). El máximo llega a
  casi 999 millones de COP, frente a una mediana de ~306 mil — hay outliers extremos además de los
  negativos.
- peso_pedido_kg físicamente imposible (mayor a 100 kg para productos individuales, incluyendo
  casos como una camisa de lino "pesando" hasta 1,400 kg): 32,045 registros (5.01%). Todo apunta a
  un error de escritura (posible corrimiento de decimal o mezcla de unidades), no a envíos reales.
- edad_cliente fuera de rango físico válido (negativa o mayor a 100 años): 27,829 registros
  (4.35%).
- Regla de dominio: fecha_pedido no puede ser posterior a la fecha de referencia (2026-08-04) →
  16,485 registros (2.58%) tienen una fecha de compra en el futuro.
- Chequeo de regla de negocio (no es un problema, es una validación positiva): monto_compra =
  precio_unitario × unidades_vendidas se cumple en el 100% de los 618,360 registros donde ambos
  campos están presentes, así que monto_compra sí es un campo derivado confiable.

## 3. Consistencia

Mide si un mismo concepto se representa siempre de la misma forma. Se normalizan mayúsculas y
minúsculas, espacios en blanco sobrantes y errores de codificación de caracteres (tildes/eñes
corrompidas) en canal_compra, metodo_pago, categoria_producto, nivel_satisfaccion y ciudad_tienda
(esta última incluye además abreviaturas de ciudad como BOG o MDE), y se contrasta el departamento
implícito en codigo_postal contra la ciudad declarada.

In [10]:
def normaliza_texto(s):
    # Quita tildes/errores de encoding, espacios sobrantes y mayusculas para comparar variantes
    if pd.isna(s):
        return s
    s2 = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return s2.strip().lower()

# --- canal_compra: 20 valores unicos -> 4 canales reales (mayusc/minusc + espacios sobrantes) ---
canal_norm = df['canal_compra'].apply(normaliza_texto)
canal_canonico_ok = df['canal_compra'].isin(['App', 'Web', 'Tienda', 'Marketplace'])
canal_requiere_norm = (~canal_canonico_ok).sum()
print("canal_compra: valores unicos crudos =", df['canal_compra'].nunique(), "-> canales reales =", canal_norm.nunique())
print(f"canal_compra: registros que requieren normalizacion: {canal_requiere_norm:,} ({canal_requiere_norm/N*100:.2f}%)")

canal_compra: valores unicos crudos = 20 -> canales reales = 4
canal_compra: registros que requieren normalizacion: 154,781 (24.18%)


In [11]:
# --- metodo_pago: 20 valores unicos -> 4 metodos reales ---
metodo_norm = df['metodo_pago'].apply(normaliza_texto)
metodo_canonico_ok = df['metodo_pago'].isin(['Efectivo', 'Tarjeta', 'PayPal', 'Transferencia'])
metodo_requiere_norm = (~metodo_canonico_ok).sum()
print("metodo_pago: valores unicos crudos =", df['metodo_pago'].nunique(), "-> metodos reales =", metodo_norm.nunique())
print(f"metodo_pago: registros que requieren normalizacion: {metodo_requiere_norm:,} ({metodo_requiere_norm/N*100:.2f}%)")

metodo_pago: valores unicos crudos = 20 -> metodos reales = 4
metodo_pago: registros que requieren normalizacion: 129,372 (20.21%)


In [12]:
# --- categoria_producto: 32 valores unicos -> 6 categorias reales ---
categoria_norm = df['categoria_producto'].apply(normaliza_texto)
categoria_canonico_ok = df['categoria_producto'].isin(['Belleza', 'Electronica', 'Deportes', 'Hogar', 'Jugueteria', 'Moda'])
categoria_requiere_norm = (~categoria_canonico_ok).sum()
print("categoria_producto: valores unicos crudos =", df['categoria_producto'].nunique(), "-> categorias reales =", categoria_norm.nunique())
print(f"categoria_producto: registros que requieren normalizacion: {categoria_requiere_norm:,} ({categoria_requiere_norm/N*100:.2f}%)")

categoria_producto: valores unicos crudos = 32 -> categorias reales = 6
categoria_producto: registros que requieren normalizacion: 416,457 (65.07%)


In [13]:
# --- nivel_satisfaccion: 15 valores unicos -> 3 niveles reales ---
nivel_norm = df['nivel_satisfaccion'].apply(normaliza_texto)
nivel_canonico_ok = df['nivel_satisfaccion'].isin(['Alto', 'Medio', 'Bajo'])
nivel_requiere_norm = ((~nivel_canonico_ok) & df['nivel_satisfaccion'].notna()).sum()
print("nivel_satisfaccion: valores unicos crudos =", df['nivel_satisfaccion'].nunique(), "-> niveles reales =", nivel_norm.nunique())
print(f"nivel_satisfaccion: registros que requieren normalizacion: {nivel_requiere_norm:,} ({nivel_requiere_norm/N*100:.2f}%)")

nivel_satisfaccion: valores unicos crudos = 15 -> niveles reales = 3
nivel_satisfaccion: registros que requieren normalizacion: 95,706 (14.95%)


In [14]:
# --- ciudad_tienda: 88 valores unicos -> 12 ciudades reales, incluyendo abreviaturas
# (BOG, MDE, CTG, BAQ, BGA, MZL, SMR), sufijos ("Bogota D.C.", "Medellin, Ant.") y variantes
# de escritura ("B/quilla", "Santiago de Cali", "Cartagena de Indias") ---
abreviaturas = {
    'ctg': 'cartagena', 'baq': 'barranquilla', 'bga': 'bucaramanga', 'mde': 'medellin',
    'mzl': 'manizales', 'smr': 'santa marta', 'bog': 'bogota',
}

def normaliza_ciudad(s):
    if pd.isna(s):
        return s
    s2 = normaliza_texto(s).replace(',', ' ')
    for sufijo in [' d.c.', ' dc', ' ant.', ' caldas']:
        s2 = s2.replace(sufijo, '')
    s2 = s2.replace('b/quilla', 'barranquilla').replace('b/manga', 'bucaramanga')
    s2 = s2.replace('santiago de cali', 'cali').replace('cartagena de indias', 'cartagena')
    s2 = s2.replace('santamarta', 'santa marta').replace('sta. marta', 'santa marta').replace('sta marta', 'santa marta')
    s2 = ' '.join(s2.split())
    return abreviaturas.get(s2, s2)

ciudad_norm = df['ciudad_tienda'].apply(normaliza_ciudad)
ciudades_canonicas = ['Bogota', 'Medellin', 'Cali', 'Barranquilla', 'Cartagena', 'Bucaramanga',
                       'Pereira', 'Manizales', 'Ibague', 'Cucuta', 'Villavicencio', 'Santa Marta']
ciudad_canonico_ok = df['ciudad_tienda'].isin(ciudades_canonicas)
ciudad_requiere_norm = ((~ciudad_canonico_ok) & df['ciudad_tienda'].notna()).sum()
print("ciudad_tienda: valores unicos crudos =", df['ciudad_tienda'].nunique(), "-> ciudades reales =", ciudad_norm.nunique())
print(f"ciudad_tienda: registros que requieren normalizacion: {ciudad_requiere_norm:,} ({ciudad_requiere_norm/N*100:.2f}%)")
print(ciudad_norm.value_counts(dropna=False))

ciudad_tienda: valores unicos crudos = 88 -> ciudades reales = 12
ciudad_tienda: registros que requieren normalizacion: 391,920 (61.24%)
ciudad_tienda
cartagena        51079
cali             51002
manizales        50991
barranquilla     50884
pereira          50863
bucaramanga      50840
bogota           50810
medellin         50775
santa marta      50755
cucuta           50735
villavicencio    50702
ibague           50636
NaN              29928
Name: count, dtype: int64


In [15]:
# --- consistencia cruzada: departamento implicito en codigo_postal vs ciudad_tienda declarada ---
# codigo_postal sigue el patron DIVIPOLA: 2 digitos de departamento + 3 de municipio + 2 de zona.
# Si falta, se rellena a la izquierda con cero (los departamentos 01-09 pierden el cero al
# guardarse como entero), y se compara el departamento resultante contra el que le corresponde
# a la ciudad declarada.
departamento_esperado_por_ciudad = {
    'bogota': '11', 'medellin': '05', 'cali': '76', 'barranquilla': '08', 'cartagena': '13',
    'bucaramanga': '68', 'cucuta': '54', 'ibague': '73', 'manizales': '17', 'pereira': '66',
    'santa marta': '47', 'villavicencio': '50',
}

codigo_postal_str = df['codigo_postal'].astype(str).str.zfill(7)
departamento_extraido = codigo_postal_str.str[:2]
departamento_esperado = ciudad_norm.map(departamento_esperado_por_ciudad)

mask_ciudad_conocida = ciudad_norm.notna()
departamento_inconsistente = ((departamento_extraido != departamento_esperado) & mask_ciudad_conocida).sum()
print(f"codigo_postal con departamento inconsistente respecto a ciudad_tienda: {departamento_inconsistente:,} ({departamento_inconsistente/N*100:.2f}%)")

codigo_postal con departamento inconsistente respecto a ciudad_tienda: 42,405 (6.63%)


Hallazgos — Consistencia (sobre 640,000 registros):

- categoria_producto: 32 valores únicos representan en realidad solo 6 categorías (Belleza,
  Electrónica, Deportes, Hogar, Juguetería, Moda) → 416,457 registros (65.07%) requieren
  normalización.
- ciudad_tienda: 88 valores únicos representan 12 ciudades reales, mezcladas por
  mayúsculas/minúsculas, codificación de caracteres rota, abreviaturas de 3 letras (BOG, MDE, CTG,
  BAQ, BGA, MZL, SMR) y sufijos administrativos ("Bogotá D.C.", "Medellín, Ant.") → 391,920
  registros (61.24%) no están en su forma canónica limpia.
- canal_compra: 20 valores únicos representan 4 canales reales (App, Web, Tienda, Marketplace); el
  problema no es solo mayúsculas/minúsculas sino también espacios en blanco al inicio o al final
  del valor (" Tienda", "App ") → 154,781 registros (24.18%) requieren normalización.
- metodo_pago: mismo patrón que canal_compra, 20 valores únicos para 4 métodos reales (Efectivo,
  Tarjeta, PayPal, Transferencia) → 129,372 registros (20.21%) requieren normalización.
- nivel_satisfaccion: 15 valores únicos representan 3 niveles reales (Alto/Medio/Bajo) → 95,706
  registros (14.95%) requieren normalización.
- Consistencia cruzada codigo_postal ↔ ciudad_tienda: extrayendo el código de departamento
  implícito en codigo_postal (formato DIVIPOLA) y comparándolo contra el departamento real de la
  ciudad declarada, 42,405 registros (6.63%) son inconsistentes — el código postal no corresponde
  al departamento de la ciudad de la tienda.

## 4. Validez

Mide si los valores presentes respetan el formato/dominio esperado para su tipo de dato, más allá de
si coinciden entre sí (eso ya se cubrió en consistencia). Se valida el formato de correo_cliente, el
formato de fecha de fecha_pedido y el formato numérico de codigo_postal.

In [16]:
# --- correo_cliente: formato basico usuario@dominio.tld ---
patron_correo = re.compile(r'^[^@\s]+@[^@\s]+\.[a-zA-Z]{2,}$')
correo_no_nulo = df['correo_cliente'].notna()
correo_valido = df['correo_cliente'].str.match(patron_correo, na=False)
correo_formato_invalido = (correo_no_nulo & ~correo_valido).sum()

print(f"correo_cliente no nulos: {correo_no_nulo.sum():,}")
print(f"correo_cliente con formato invalido (de los no nulos): {correo_formato_invalido:,} ({correo_formato_invalido/N*100:.2f}% del total)")
print("Ejemplos:", df.loc[correo_no_nulo & ~correo_valido, 'correo_cliente'].head(5).tolist())

correo_cliente no nulos: 534,688
correo_cliente con formato invalido (de los no nulos): 51,532 (8.05% del total)
Ejemplos: ['cliente593809.hotmail.com', 'cliente583717.outlook.com', 'cliente232336.gmail.com', 'cliente184801.gmail.com', 'cliente427463.hotmail.com']


In [17]:
# --- fecha_pedido: solo se aceptan 2 formatos (YYYY-MM-DD o DD/MM/YYYY); todo lo demas o
# fechas de calendario inexistentes (ej. 30/02/2025) se marca como formato invalido ---
fecha_pedido_formato_invalido = fecha_pedido.isna().sum()
print(f"fecha_pedido con formato invalido o fecha inexistente: {fecha_pedido_formato_invalido:,} ({fecha_pedido_formato_invalido/N*100:.2f}%)")
print("Ejemplos de valores no parseables:")
print(df.loc[fecha_pedido.isna(), 'fecha_pedido'].drop_duplicates().head(5).tolist())

fecha_pedido con formato invalido o fecha inexistente: 22,687 (3.54%)
Ejemplos de valores no parseables:
['30/02/2025', '00/00/0000', '2026-13-05', '31/04/2025']


In [18]:
# --- codigo_postal: el formato DIVIPOLA esperado son 7 digitos (departamento 2 + municipio 3
# + zona 2). Los departamentos de un solo digito (Antioquia=05, Atlantico=08) pierden el cero
# inicial al guardarse como numero entero y quedan en 6 digitos ---
largo_cp = df['codigo_postal'].astype(str).str.len()
print("Distribucion de longitud de codigo_postal (digitos):")
print(largo_cp.value_counts().sort_index())

cp_formato_invalido = (largo_cp != 7).sum()
print(f"\ncodigo_postal que NO tiene 7 digitos (formato esperado): {cp_formato_invalido:,} ({cp_formato_invalido/N*100:.2f}%)")

# confirmacion: el problema se concentra en Medellin y Barranquilla (depto de 1 digito)
print(pd.crosstab(ciudad_norm, largo_cp))

Distribucion de longitud de codigo_postal (digitos):


codigo_postal
6    106725
7    533275
Name: count, dtype: int64

codigo_postal que NO tiene 7 digitos (formato esperado): 106,725 (16.68%)


codigo_postal      6      7
ciudad_tienda              
barranquilla   47745   3139
bogota           598  50212
bucaramanga      629  50211
cali             668  50334
cartagena        662  50417
cucuta           650  50085
ibague           616  50020
manizales        652  50339
medellin       47588   3187
pereira          665  50198
santa marta      649  50106
villavicencio    609  50093


Hallazgos — Validez (sobre 640,000 registros):

- codigo_postal: el formato DIVIPOLA esperado tiene 7 dígitos (2 de departamento + 3 de municipio +
  2 de zona). 106,725 registros (16.68%) tienen solo 6 dígitos porque perdieron el cero inicial del
  código de departamento al guardarse como número entero — esto se concentra casi por completo en
  Medellín (Antioquia = "05") y Barranquilla (Atlántico = "08"), los dos departamentos de código de
  un solo dígito entre las 12 ciudades del dataset.
- correo_cliente: de los 534,688 no nulos, 51,532 registros (8.05% del total) no tienen formato de
  correo válido (p.ej. "cliente73207.outlook.com", sin el símbolo @).
- fecha_pedido: 22,687 registros (3.54%) no son parseables bajo los dos formatos aceptados
  (YYYY-MM-DD o DD/MM/YYYY) o corresponden a fechas de calendario inexistentes, como "30/02/2025" o
  "2026-13-05" (no existe el mes 13).

## 5. Unicidad

Mide si cada entidad (en este caso, cada pedido) está representada una sola vez. Se revisan
duplicados de fila completa y duplicados sobre el subset correcto: id_pedido, que debería
funcionar como llave primaria de la tabla.

In [19]:
duplicados_fila_completa = df.duplicated().sum()
print(f"Filas 100% duplicadas (todas las columnas identicas): {duplicados_fila_completa:,} ({duplicados_fila_completa/N*100:.2f}%)")

# subset correcto: id_pedido deberia ser unico por pedido
id_pedido_duplicado_filas = df.duplicated(subset='id_pedido', keep=False).sum()
id_pedido_duplicado_pedidos = df['id_pedido'].duplicated().sum()
print(f"Filas con id_pedido duplicado: {id_pedido_duplicado_filas:,} ({id_pedido_duplicado_filas/N*100:.2f}%)")
print(f"Pedidos (id_pedido) que aparecen mas de una vez: {id_pedido_duplicado_pedidos:,}")

df[df.duplicated(subset='id_pedido', keep=False)].sort_values('id_pedido').head(6)

Filas 100% duplicadas (todas las columnas identicas): 20,000 (3.12%)


Filas con id_pedido duplicado: 40,000 (6.25%)
Pedidos (id_pedido) que aparecen mas de una vez: 20,000


,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,unidades_vendidas,monto_compra,edad_cliente,correo_cliente,nivel_satisfaccion,nivel_lealtad,comentario_cliente,peso_pedido_kg,fecha_actualizacion_stock,fuga_cliente
172693,P100005,C164871,2025-01-28,Tienda,Transferencia,MDE,500101,Juguetería,Rompecabezas 1000 piezas,595200.0,-1,-595200.0,40,cliente5@yahoo.es,NaN,Platino,Rápido y sin problemas,13.0,2026-02-28,0
174646,P100005,C164871,2025-01-28,Tienda,Transferencia,MDE,500101,Juguetería,Rompecabezas 1000 piezas,595200.0,-1,-595200.0,40,cliente5@yahoo.es,NaN,Platino,Rápido y sin problemas,13.0,2026-02-28,0
418817,P100010,C76626,17/8/2025,Web,Transferencia,SANTA MARTA,4700103,Hogar,Olla a presión,520000.0,1,520000.0,52,cliente10@novamarket.com.co,alto,Bronce,El producto llegó dañado,8.5,2026-04-15,0
368632,P100010,C76626,17/8/2025,Web,Transferencia,SANTA MARTA,4700103,Hogar,Olla a presión,520000.0,1,520000.0,52,cliente10@novamarket.com.co,alto,Bronce,El producto llegó dañado,8.5,2026-04-15,0
168620,P100023,C46545,00/00/0000,App,Efectivo,Cúcuta,5400110,Belleza,Secador de pelo,271800.0,1,271800.0,64,cliente23.outlook.com,Bajo,Plata,Cumplió con lo prometido,10.1,2026-04-02,1
103928,P100023,C46545,00/00/0000,App,Efectivo,Cúcuta,5400110,Belleza,Secador de pelo,271800.0,1,271800.0,64,cliente23.outlook.com,Bajo,Plata,Cumplió con lo prometido,10.1,2026-04-02,1


Hallazgos — Unicidad (sobre 640,000 registros):

- 20,000 filas (3.12%) están completamente duplicadas: mismo id_pedido con exactamente los mismos
  valores en las 20 columnas. Cada una infla directamente el conteo de pedidos, unidades vendidas e
  ingresos totales si no se deduplica.
- Usando el subset correcto (id_pedido, la llave que debería ser primaria): 40,000 registros
  (6.25%), es decir 20,000 pedidos distintos, aparecen más de una vez. En su estado actual
  id_pedido no puede usarse como llave para hacer join o para contar pedidos únicos sin antes
  deduplicar.

## 6. Oportunidad

Mide qué tan actualizados/vigentes están los datos respecto al momento en que se consultan.
fecha_actualizacion_stock es la única columna que representa directamente la "vigencia" de un
dato (el inventario). Se define un umbral de 90 días: NovaMarket vende categorías de alta rotación
(moda, belleza, electrónica), por lo que un registro de stock con más de un trimestre de
antigüedad ya no refleja el inventario real y se considera desactualizado.

In [20]:
UMBRAL_DIAS = 90
fecha_stock = pd.to_datetime(df['fecha_actualizacion_stock'], format='%Y-%m-%d', errors='coerce')
print("fecha_actualizacion_stock no parseable:", fecha_stock.isna().sum())

limite_vigencia = HOY - pd.Timedelta(days=UMBRAL_DIAS)
stock_desactualizado = (fecha_stock < limite_vigencia).sum()
stock_fecha_futura = (fecha_stock > HOY).sum()

print(f"Umbral de vigencia: {UMBRAL_DIAS} dias (limite: {limite_vigencia.date()}, hoy: {HOY.date()})")
print(f"Registros de stock desactualizados (> {UMBRAL_DIAS} dias): {stock_desactualizado:,} ({stock_desactualizado/N*100:.2f}%)")
print(f"Registros de stock con fecha futura (imposible): {stock_fecha_futura:,} ({stock_fecha_futura/N*100:.2f}%)")
print()
print(fecha_stock.describe())

fecha_actualizacion_stock no parseable: 0
Umbral de vigencia: 90 dias (limite: 2026-05-06, hoy: 2026-08-04)
Registros de stock desactualizados (> 90 dias): 327,601 (51.19%)
Registros de stock con fecha futura (imposible): 21,722 (3.39%)

count                        640000
mean     2026-02-24 01:43:19.335000
min             2023-01-01 00:00:00
25%             2026-03-15 00:00:00
50%             2026-05-03 00:00:00
75%             2026-06-21 00:00:00
max             2027-12-28 00:00:00
Name: fecha_actualizacion_stock, dtype: object


Hallazgos — Oportunidad (sobre 640,000 registros, umbral de 90 días justificado por la alta
rotación de las categorías de producto de NovaMarket):

- 327,601 registros (51.19%) tienen su última actualización de stock con más de 90 días de
  antigüedad respecto al 2026-08-04. Es, de lejos, el hallazgo más grande de toda la auditoría: más
  de la mitad del inventario reportado no refleja la realidad actual.
- 21,722 registros (3.39%) tienen fecha_actualizacion_stock en una fecha futura, lo cual es
  físicamente imposible y evidencia un error de captura, no solo de desactualización.

## 7. Catálogo de problemas

Se consolidan los hallazgos de las 6 dimensiones en un catálogo único. La severidad de cada fila
se asigna combinando dos criterios, tal como pide la rúbrica:

1. % de registros afectados.
2. Criticidad de la columna para el negocio: precio_unitario, monto_compra, unidades_vendidas,
   id_pedido y fecha_pedido son de criticidad alta (facturación, trazabilidad de pedidos y
   reporting temporal); correo_cliente, ciudad_tienda, categoria_producto, nivel_satisfaccion,
   edad_cliente, canal_compra, metodo_pago, peso_pedido_kg y fecha_actualizacion_stock son de
   criticidad media (segmentación, CRM, logística, inventario); codigo_postal y comentario_cliente
   son de criticidad baja (referencia geográfica secundaria y texto libre opcional).

Regla de severidad aplicada:
- Columna de criticidad alta → Alta si % ≥ 3%, si no Media.
- Columna de criticidad media → Alta si % ≥ 15%, Media si 5% ≤ % < 15%, Baja si % < 5%.
- Columna de criticidad baja → tope en Media (nunca Alta, no compromete operación crítica); Media
  si % ≥ 50%, si no Baja.
- Problemas de unicidad (duplicados) → siempre Alta, porque distorsionan cualquier métrica
  agregada (ingresos, pedidos, unidades).

In [21]:
catalogo = pd.DataFrame([
    # dimension, columna_afectada, problema, cantidad_afectada, severidad
    ('Completitud', 'correo_cliente', 'Correo electronico faltante', 105312, 'Alta'),
    ('Completitud', 'precio_unitario / monto_compra', 'Precio y monto faltantes (mismas filas)', 21640, 'Alta'),
    ('Completitud', 'nivel_satisfaccion', 'Nivel de satisfaccion faltante', 46469, 'Media'),
    ('Completitud', 'ciudad_tienda', 'Ciudad de tienda faltante', 29928, 'Baja'),
    ('Completitud', 'comentario_cliente', 'Comentario faltante (campo opcional de texto libre)', 90806, 'Baja'),

    ('Exactitud', 'unidades_vendidas', 'Unidades negativas, en cero o con valores centinela (5000 / 9999)', 46403, 'Alta'),
    ('Exactitud', 'precio_unitario', 'Precio con valor negativo (imposible)', 19629, 'Alta'),
    ('Exactitud', 'peso_pedido_kg', 'Peso fisicamente imposible (> 100 kg para un producto individual)', 32045, 'Media'),
    ('Exactitud', 'edad_cliente', 'Edad fuera de rango fisico valido (negativa o mayor a 100 anios)', 27829, 'Baja'),
    ('Exactitud', 'fecha_pedido', 'Fecha de pedido posterior a hoy (regla de dominio: no puede ser futura)', 16485, 'Media'),

    ('Consistencia', 'categoria_producto', 'Variantes de mayusc/minusc y codificacion (32 valores -> 6 categorias reales)', 416457, 'Alta'),
    ('Consistencia', 'ciudad_tienda', 'Variantes, abreviaturas y codificacion (88 valores -> 12 ciudades reales)', 391920, 'Alta'),
    ('Consistencia', 'canal_compra', 'Variantes de mayusc/minusc y espacios sobrantes (20 valores -> 4 canales reales)', 154781, 'Alta'),
    ('Consistencia', 'metodo_pago', 'Variantes de mayusc/minusc y espacios sobrantes (20 valores -> 4 metodos reales)', 129372, 'Alta'),
    ('Consistencia', 'nivel_satisfaccion', 'Variantes de mayusc/minusc (15 valores -> 3 niveles reales)', 95706, 'Media'),
    ('Consistencia', 'codigo_postal / ciudad_tienda', 'Departamento implicito en codigo_postal inconsistente con la ciudad declarada', 42405, 'Media'),

    ('Validez', 'codigo_postal', 'No tiene los 7 digitos DIVIPOLA esperados (perdida de cero inicial en depto. de 1 digito)', 106725, 'Baja'),
    ('Validez', 'correo_cliente', "Formato de correo invalido (sin '@' o sin dominio)", 51532, 'Media'),
    ('Validez', 'fecha_pedido', 'Formato de fecha invalido o fecha de calendario inexistente (ej. 30/02/2025)', 22687, 'Alta'),

    ('Unicidad', 'id_pedido (fila completa)', 'Filas 100% duplicadas (todas las columnas identicas)', 20000, 'Alta'),
    ('Unicidad', 'id_pedido', 'id_pedido repetido: no funciona como llave primaria (20,000 pedidos duplicados)', 40000, 'Alta'),

    ('Oportunidad', 'fecha_actualizacion_stock', 'Stock desactualizado: mas de 90 dias de antiguedad respecto al 2026-08-04', 327601, 'Alta'),
    ('Oportunidad', 'fecha_actualizacion_stock', 'Fecha de actualizacion de stock futura (posterior a hoy, imposible)', 21722, 'Baja'),
], columns=['dimension', 'columna_afectada', 'problema', 'cantidad_afectada', 'severidad'])

catalogo['porcentaje_afectado'] = (catalogo['cantidad_afectada'] / N * 100).round(2)

orden_severidad = {'Alta': 3, 'Media': 2, 'Baja': 1}
catalogo['_orden'] = catalogo['severidad'].map(orden_severidad)
catalogo = catalogo.sort_values(['_orden', 'porcentaje_afectado'], ascending=[False, False]).drop(columns='_orden').reset_index(drop=True)

catalogo = catalogo[['dimension', 'columna_afectada', 'problema', 'cantidad_afectada', 'porcentaje_afectado', 'severidad']]
catalogo

,dimension,columna_afectada,problema,cantidad_afectada,porcentaje_afectado,severidad
0,Consistencia,categoria_producto,Variantes de mayusc/minusc y codificacion (32 ...,416457,65.07,Alta
1,Consistencia,ciudad_tienda,"Variantes, abreviaturas y codificacion (88 val...",391920,61.24,Alta
2,Oportunidad,fecha_actualizacion_stock,Stock desactualizado: mas de 90 dias de antigu...,327601,51.19,Alta
3,Consistencia,canal_compra,Variantes de mayusc/minusc y espacios sobrante...,154781,24.18,Alta
4,Consistencia,metodo_pago,Variantes de mayusc/minusc y espacios sobrante...,129372,20.21,Alta
5,Completitud,correo_cliente,Correo electronico faltante,105312,16.46,Alta
6,Exactitud,unidades_vendidas,"Unidades negativas, en cero o con valores cent...",46403,7.25,Alta
7,Unicidad,id_pedido,id_pedido repetido: no funciona como llave pri...,40000,6.25,Alta
8,Validez,fecha_pedido,Formato de fecha invalido o fecha de calendari...,22687,3.54,Alta
9,Completitud,precio_unitario / monto_compra,Precio y monto faltantes (mismas filas),21640,3.38,Alta


In [22]:
catalogo.to_csv('S03_PD_Pallares_CatalogoProblemas.csv', index=False, encoding='utf-8-sig')
print(f"Catalogo guardado: S03_PD_Pallares_CatalogoProblemas.csv ({len(catalogo)} filas)")
print("\nDistribucion de severidad:")
print(catalogo['severidad'].value_counts())

Catalogo guardado: S03_PD_Pallares_CatalogoProblemas.csv (23 filas)

Distribucion de severidad:
severidad
Alta     12
Media     6
Baja      5
Name: count, dtype: int64


## 8. Conclusión

El dataset crudo de NovaMarket no es confiable para reportar al comité sin un proceso de limpieza
previo. Con 640,000 registros, los problemas más graves y de mayor impacto en el negocio son:

1. Oportunidad: más de la mitad de los registros (51.19%) tiene el stock desactualizado por más de
   90 días — el hallazgo individual más grande de toda la auditoría.
2. Consistencia de texto (categoria_producto, ciudad_tienda, canal_compra, metodo_pago): entre 20%
   y 65% de los registros, según la columna, necesita normalización antes de poder agregar o
   segmentar correctamente. Este problema no es solo de mayúsculas/minúsculas: incluye espacios en
   blanco sobrantes, abreviaturas de ciudad y errores de codificación de caracteres.
3. Duplicados: 20,000 pedidos duplicados inflan cualquier métrica agregada de ingresos o unidades.
4. unidades_vendidas y precio_unitario con valores imposibles (negativos, cero o centinela)
   comprometen directamente el cálculo de ingresos, aunque la buena noticia es que monto_compra
   sí es 100% consistente con precio_unitario × unidades_vendidas cuando ambos están presentes.

Antes de cualquier análisis o reporte para el comité, se recomienda: deduplicar por id_pedido,
normalizar texto (mayúsculas/minúsculas, espacios y encoding) en categoria_producto, ciudad_tienda,
canal_compra, metodo_pago y nivel_satisfaccion, descartar o corregir registros con
precio_unitario/unidades_vendidas/edad_cliente/peso_pedido_kg imposibles, estandarizar el parseo de
fecha_pedido a un único formato, y priorizar la actualización del stock dado que más de la mitad de
los registros de inventario está desactualizado.